In [ ]:
import os
from kaggle_secrets import UserSecretsClient

# 1) Add your PAT as a Kaggle secret: Settings → Secrets → "MY_GITHUB_TOKEN"
#    (must have repo + workflow scope so the pipeline can push artifacts)
os.environ["GITHUB_TOKEN"] = UserSecretsClient().get_secret("MY_GITHUB_TOKEN")

# 2) Fresh clone into the Kaggle working dir (NOT /content — that's Colab)
!rm -rf sports_prediction_model
!git clone -q https://github.com/andrewkemmer/sports_prediction_model.git sports_prediction_model

# 3) Install every dependency the pipeline imports (backend, not the Streamlit frontend)
!pip install -q pybaseball duckdb shap optuna \
    lightgbm xgboost scikit-learn pandas numpy joblib requests
print("setup done")



import os

# --- OPTIONAL overrides (omit everything for a normal daily run) ---

# Custom window: only set these for one-off backfills, NOT daily runs
os.environ["MLB_START_DATE"] = "2024-01-01"
os.environ["MLB_END_DATE"]   = "2026-09-02"   # omit = runs through today

# Full repull: only for a rebuild; omit for daily (chunk cache handles incremental)
os.environ["MLB_FULL_REPULL"] = "1"   # set once, then remove

print("run options set")



import os
import subprocess

repo = "/kaggle/working/sports_prediction_model"
cmd = ["python", "mlb-backend/backend/master_pipeline.py"]


def _sh(c):
    return subprocess.run(c, cwd=repo, capture_output=True, text=True).stdout.strip()


# Capture the PRE-RUN remote tip. The verification below needs a baseline:
# `git log -1` alone is not a witness, because Phase 5/6 commit through
# GitPython and do not reliably move the local branch ref. On 2026-09-26 this
# cell printed the PRE-run SHA (f5ca785) while the ls-tree check in the very
# same cell listed the artifacts that run had just pushed.
_sh(["git", "fetch", "origin", "-q"])
PRE_RUN_TIP = _sh(["git", "rev-parse", "origin/main"])
print("Pre-run origin/main:", PRE_RUN_TIP)

result = subprocess.run(cmd, cwd=repo, env=os.environ.copy(), capture_output=False)

# The pipeline's Phase 5 already pushes data_delivery/ to GitHub itself.
# Fail loudly if it didn't complete - Kaggle marks the run failed.
if result.returncode != 0:
    raise SystemExit(f"Pipeline failed with exit code {result.returncode}")
print("Pipeline completed - artifacts pushed to GitHub by Phase 5 sync.")


# -- Verify the push against the REMOTE tip, not local HEAD --------------------
# Phase 5 has its own gate (verify_pushed_paths); this is the independent
# cross-check, so it must read origin/main and prove it advanced past the
# pre-run baseline. Reading local HEAD cannot distinguish "push landed" from
# "push silently failed".
_sh(["git", "fetch", "origin", "-q"])
tip = _sh(["git", "rev-parse", "origin/main"])
print("origin/main after run:", tip)
print("origin/main tip subject:", _sh(["git", "log", "-1", "--format=%h %s", "origin/main"]))

if tip == PRE_RUN_TIP:
    raise SystemExit(
        f"VERIFY FAILED: origin/main did not move (still {tip}). "
        "Phase 5 pushed nothing - treat this run as UNSYNCED.")
print(f"VERIFY OK: origin/main advanced {PRE_RUN_TIP[:8]} -> {tip[:8]}")

# Phase 6 commits its cleanup pass separately; surface every commit this run
# added so that pass stays visible even when the captured run log truncates
# before it (it has, twice).
print("Commits this run added to main:")
print(_sh(["git", "log", "--oneline", f"{PRE_RUN_TIP}..{tip}"]) or "  (none)")

# Sanity: newest dated artifact on main
ls = _sh(["git", "ls-tree", "-r", "--name-only", "origin/main"])
datelated = [f for f in ls.splitlines() if "mlb-backend/data_delivery/" in f and "_2026" in f and not f.rsplit("/", 1)[-1].startswith("~$")]
print("Latest dated artifacts:", sorted(datelated)[-3:] if datelated else "none found")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.1/426.1 kB 5.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.6/455.6 kB 9.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 21.2 MB/s eta 0:00:0000:01
setup done
run options set
